# Coverage matching: MEG vs iEEG

Load a dataset, inspect its variance and coverage, compute PCA, and compare it with iEEG.

MEG types: `full_average`, `full_concatenated`, `coverage_average`, `paired_coverage`, `random_control`.
The random control keeps the paired participants, electrode counts and repeated-source multiplicities, but randomises source locations.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import os 
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


# Run from iEEGvsMEG, or from its parent repository directory.
ROOT = Path.cwd()
if not (ROOT / 'utils_updated.py').exists() and (ROOT / 'iEEGvsMEG' / 'utils_updated.py').exists():
    ROOT = ROOT / 'iEEGvsMEG'
if not (ROOT / 'utils_updated.py').exists():
    raise FileNotFoundError('Set ROOT to the directory containing coverage_matching.ipynb and utils_updated.py.')
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
sys.path.insert(0, str(ROOT.parent / 'LB'))
from src.setting import GetInfo, PROJECT_PATH
from utils_updated import (
    load_dataset, compute_variance, show_coverage, compute_pca,
    plot_pca_timecourses, plot_pca_weights,
    correlate_timecourses, correlate_weights,
)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

## Participants and coordinates

In [ ]:
MEG_DIR = ROOT / 'MEG' / 'dataMEG'
IEEG_DIR = ROOT / 'ieeg_shortWOBS_fs250'

meg_subj_list = sorted(file.removesuffix('_source.p') for file in os.listdir(MEG_DIR) if file.endswith('_source.p'))
print('MEG Subject number : ', len(meg_subj_list))

ieeg_subj_list = sorted(file.removesuffix('_epochs.p') for file in os.listdir(IEEG_DIR) if file.endswith('_epochs.p'))
if not meg_subj_list or not ieeg_subj_list:
    raise FileNotFoundError('Check MEG_DIR and IEEG_DIR: no source or epoch files found.')
print('iEEG Subject number : ', len(ieeg_subj_list))

info_file = IEEG_DIR / f'{ieeg_subj_list[0]}_info.json'
with open(info_file) as json_data:
    d = json.load(json_data)
    time = d['time_epoch']

SFREQ = 250
CONDITIONS = (1, 2)    

CONDITION_MODE = 'average'  # 'average' (legacy) or 'stack'
PAIRING = None  # optional fixed dict: {'iEEG_subject': 'MEG_subject', ...}
N_COMPONENTS = 10

project_path = ROOT.parent / PROJECT_PATH
coord, areas, elect_list, subj_list, regions_ieeg = GetInfo(ieeg_subj_list, data_path=IEEG_DIR, project_path=project_path)
coord=np.array(coord)
coord = np.where(abs(coord) >100, coord/1000, coord)
coord = np.where(abs(coord) >100, coord/1000, coord)
coord = coord/1000

## Load the datasets

The first call reads the files once. Later calls reuse them through `reference=ieeg` and construct only the requested MEG setup. The default seed keeps participant pairing identical across setups.

Your prepared `coord` is in metres. Input preprocessing follows the existing pipeline: MEG channel z-score and iEEG ×1000, before averaging conditions. `meg_tmin=time[0]` assumes a shared epoch origin; replace it with `meg_times_file=...` if an independent MEG time vector is available.

In [ ]:
electrode_metadata = pd.DataFrame(coord, columns=['x', 'y', 'z'])
electrode_metadata['subject'] = subj_list
electrode_metadata['channel'] = elect_list
electrode_metadata['region'] = regions_ieeg
electrode_metadata['channel_index'] = electrode_metadata.groupby('subject', sort=False).cumcount()

ieeg = load_dataset(
    'ieeg', meg_dir=MEG_DIR, ieeg_dir=IEEG_DIR,
    meg_subjects=meg_subj_list, ieeg_subjects=ieeg_subj_list,
    electrode_metadata=electrode_metadata,
    meg_coordinate_unit='m', ieeg_coordinate_unit='m',
    meg_tmin=float(time[0]), sfreq=SFREQ, conditions=CONDITIONS,
    condition_mode=CONDITION_MODE,
)

In [ ]:
meg_types = [
    'full_average', 'full_concatenated', 'coverage_average',
    'paired_coverage', 'random_control',
]
datasets = {'iEEG': ieeg}
for kind in meg_types:
    datasets[kind] = load_dataset(kind, reference=ieeg, pairing=PAIRING)

## Variance

Total variance is the sum of feature variances across the PCA observations (`ddof=1`). Mean feature variance adjusts for feature count. Values reflect the configured preprocessing; absolute MEG and iEEG variance are not in equivalent physical units.

In [ ]:
variance = pd.DataFrame({name: compute_variance(data) for name, data in datasets.items()}).T
display(variance)

## Brain coverage

Large datasets are skipped automatically. Co-located features are shown once.

In [ ]:
for data in datasets.values():
    show_coverage(data)

## Compute PCA

In [ ]:
pca = {name: compute_pca(data, n_components=N_COMPONENTS)
       for name, data in datasets.items()}

## PCA time courses and explained variance

In [ ]:
for result in pca.values():
    plot_pca_timecourses(result)

## PCA weight maps

Signed extraction weights on glass brains; component signs are arbitrary. Show the first three here (change `n_components` to show more). In full concatenation, co-located participant weights are **averaged only for display**; PCA and correlations retain the original features.

In [ ]:
for result in pca.values():
    plot_pca_weights(result, n_components=3)

## Correlation with iEEG time courses

All component pairs are compared without reordering or sign flipping. These are descriptive, in-sample Pearson correlations.

In [ ]:
time_correlations = {}
for kind in meg_types:
    time_correlations[kind] = correlate_timecourses(pca[kind], pca['iEEG'])

## Correlation with iEEG weights

Weights are compared in iEEG electrode order: nearest full-average source, paired participant's nearest full-concatenation source, or the electrode slots of the matched datasets. The random control uses random slots and has **no anatomical correspondence**. Repeated source matches remain repeated in the comparison.

In [ ]:
weight_correlations = {}
for kind in meg_types:
    weight_correlations[kind] = correlate_weights(pca[kind], pca['iEEG'])